In [ ]:
import random

# Base realistic Indian names (mix of male + female)
base_names = [
"aarav","vivaan","aditya","vihaan","arjun","sai","reyansh","krishna",
"ishaan","shaurya","advait","dhruv","kabir","lakshya","mohit","naman",
"omkar","pranav","ritvik","samar","tanmay","utkarsh","varun","yash",
"ananya","aadhya","diya","ira","myra","sara","anika","navya","aanya","pari",
"kavya","meera","naina","priya","rhea","sanya","tanvi","vaishnavi",
"ishita","jhanvi","komal","mitali","nidhi","pallavi","riddhi","shruti"
]

# Variations to simulate LLM-like diversity
prefixes = ["", "a", "vi", "ra", "ka", "sa", "an", "sh", "pr"]
suffixes = ["", "a", "an", "it", "ya", "ika", "esh", "ansh"]

dataset = set()

# Generate until we reach 1000 unique names
while len(dataset) < 1000:
    name = random.choice(prefixes) + random.choice(base_names) + random.choice(suffixes)

    # clean unrealistic repeats
    name = name.replace("aa","a").replace("vv","v")

    dataset.add(name)

dataset = list(dataset)

# Save to file
with open("TrainingNames.txt", "w") as f:
    for name in dataset:
        f.write(name + "\n")

print("Total names:", len(dataset))
print("Unique names:", len(set(dataset)))
print("Sample names:", dataset[:20])

Total names: 1000
Unique names: 1000
Sample names: ['prdhruvansh', 'anishanika', 'lakshyaika', 'aarav', 'shsamarya', 'prmyra', 'kavivanesh', 'nainansh', 'ishanesh', 'sapranavesh', 'kapallaviya', 'nidhiya', 'prpariika', 'sakabiran', 'karheaesh', 'naman', 'vivaishnaviit', 'viyashansh', 'kakomalit', 'anshauryaya']


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Load dataset
with open("TrainingNames.txt") as f:
    names = [line.strip().lower() for line in f]

# ADD START + END TOKENS
names = ["^" + name + "$" for name in names]

# Build vocabulary
chars = sorted(list(set("".join(names))))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

vocab_size = len(chars)

def encode(name):
    return [char_to_idx[c] for c in name]

# Sequence dataset
data = []
for name in names:
    seq = encode(name)
    if len(seq) > 1:
        data.append((seq[:-1], seq[1:]))

# Hyperparameters
hidden_size = 32   # reduced to avoid overfitting
learning_rate = 0.005
epochs = 5

In [ ]:
class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        x = self.embedding(x)
        out, hidden = self.rnn(x, hidden)
        out = self.fc(out)
        return out, hidden

    def init_hidden(self):
        return torch.zeros(1, 1, hidden_size)

In [ ]:
class BLSTM(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, hidden_size)

        # NO DROPOUT (since num_layers=1)
        self.lstm = nn.LSTM(
            hidden_size,
            hidden_size,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x, hidden):
        x = self.embedding(x)
        out, hidden = self.lstm(x, hidden)
        out = self.fc(out)
        return out, hidden

    def init_hidden(self):
        return (
            torch.zeros(2, 1, hidden_size),
            torch.zeros(2, 1, hidden_size)
        )

In [ ]:
class AttentionRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        x = self.embedding(x)

        out, hidden = self.rnn(x, hidden)  # (B, T, H)

        # Self-attention
        scores = torch.bmm(out, out.transpose(1, 2))
        weights = torch.softmax(scores, dim=-1)

        context = torch.bmm(weights, out)

        out = out + context
        out = self.fc(out)

        return out, hidden

    def init_hidden(self):
        return torch.zeros(1, 1, hidden_size)

In [ ]:
def train(model, name="Model", epochs_override=None):
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.CrossEntropyLoss()

    use_epochs = epochs_override if epochs_override else epochs

    for epoch in range(use_epochs):
        total_loss = 0

        for inp_seq, target_seq in data:
            inp = torch.tensor([inp_seq])
            target = torch.tensor(target_seq)

            hidden = model.init_hidden()

            optimizer.zero_grad()

            output, hidden = model(inp, hidden)
            output = output.squeeze(0)

            loss = loss_fn(output, target)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"{name} Epoch {epoch+1}, Avg Loss: {total_loss/len(data):.4f}")

In [ ]:
def generate_name(model, max_len=15, temperature=1.2, top_k=5):
    model.eval()

    char = torch.tensor([[char_to_idx["^"]]])
    hidden = model.init_hidden()

    name = ""

    for _ in range(max_len):
        output, hidden = model(char, hidden)

        logits = output[0, -1] / temperature
        probs = torch.softmax(logits, dim=0)

        # Top-k sampling
        top_probs, top_idx = torch.topk(probs, top_k)
        top_probs = top_probs / torch.sum(top_probs)

        idx = top_idx[torch.multinomial(top_probs, 1)].item()
        next_char = idx_to_char[idx]

        # Stop condition
        if next_char == "$":
            if len(name) >= 3:
                break
            else:
                continue

        # Avoid special tokens
        if next_char not in ["^", "$"]:
            name += next_char

        char = torch.tensor([[idx]])

    return name

In [ ]:
# RNN
print(generate_name(rnn_model, temperature=1.0, top_k=5))

# BLSTM (needs more randomness)
print(generate_name(blstm_model, temperature=1.5, top_k=8))

# Attention
print(generate_name(attn_model, temperature=1.2, top_k=6))

vinamohaya
alalalalalalll
kamyra


In [ ]:
# Train models
rnn_model = VanillaRNN(vocab_size, hidden_size)
train(rnn_model, "RNN")

blstm_model = BLSTM(vocab_size, hidden_size)
train(blstm_model, "BLSTM", epochs_override=2)  # early stop

attn_model = AttentionRNN(vocab_size, hidden_size)
train(attn_model, "Attention")

# Generate samples
print("\nRNN Samples:")
for _ in range(5):
    print(generate_name(rnn_model))

print("\nBLSTM Samples:")
for _ in range(5):
    print(generate_name(blstm_model))

print("\nAttention Samples:")
for _ in range(5):
    print(generate_name(attn_model))

RNN Epoch 1, Avg Loss: 1.6245
RNN Epoch 2, Avg Loss: 1.2835
RNN Epoch 3, Avg Loss: 1.1678
RNN Epoch 4, Avg Loss: 1.1057
RNN Epoch 5, Avg Loss: 1.0752
BLSTM Epoch 1, Avg Loss: 0.3039
BLSTM Epoch 2, Avg Loss: 0.0072
Attention Epoch 1, Avg Loss: 1.6208
Attention Epoch 2, Avg Loss: 1.2827
Attention Epoch 3, Avg Loss: 1.1696
Attention Epoch 4, Avg Loss: 1.1144
Attention Epoch 5, Avg Loss: 1.0847

RNN Samples:
asamyanshya
saprritvikansh
kadiyan
arjunya
anyashya

BLSTM Samples:
nananananananan
rararararararar
rrrrururrrururu
ananananananana
rararararara

Attention Samples:
raran
sapryasait
kanaman
advait
karitvik


In [ ]:
def generate_samples(model, n=200, temp=1.2, k=6):
    samples = []
    for _ in range(n):
        name = generate_name(model, temperature=temp, top_k=k)
        samples.append(name)
    return samples

In [ ]:
# Remove tokens for comparison
train_set = set([name.strip("^$") for name in names])

In [ ]:
def novelty_rate(generated, train_set):
    novel = [name for name in generated if name not in train_set]
    return len(novel) / len(generated)

In [ ]:
def diversity(generated):
    return len(set(generated)) / len(generated)

In [ ]:
# Generate samples
rnn_samples = generate_samples(rnn_model, temp=1.0, k=5)
blstm_samples = generate_samples(blstm_model, temp=1.5, k=8)
attn_samples = generate_samples(attn_model, temp=1.2, k=6)

# Compute metrics
print("RNN Novelty:", novelty_rate(rnn_samples, train_set))
print("RNN Diversity:", diversity(rnn_samples))

print("BLSTM Novelty:", novelty_rate(blstm_samples, train_set))
print("BLSTM Diversity:", diversity(blstm_samples))

print("Attention Novelty:", novelty_rate(attn_samples, train_set))
print("Attention Diversity:", diversity(attn_samples))

RNN Novelty: 0.765
RNN Diversity: 0.905
BLSTM Novelty: 1.0
BLSTM Diversity: 0.56
Attention Novelty: 0.775
Attention Diversity: 0.925
